In [5]:
%matplotlib widget

import os
import mne
import tqdm
import pytz
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from tqdm import tqdm
from pathlib import Path
from mne.io import concatenate_raws, read_raw_edf
from edf2parquet.readers import ParquetReader
from edf2parquet.converters import EdfToParquetConverter, AdvancedEdfToParquetConverter

from bscarlos.settings import RAW_KISPI_DATA_FOLDER
from bscarlos.settings import PROCESSED_KISPI_DATA_FOLDER


# ETL
Here we explore the extract the raw data (.edf) from KISPI patients and their corresponding annotations (.txt) from our `RAW_KISPI_DATA_FOLDER`.

We transform the data into more managable `pandas.dataframes` with the goal to combine the annotations which are our 'ground_truth' labels.

In the end we should have a dataframe for each patients containing 17-bipolar channel columns and 1-ground truth column.

We load/export our newly merged dataframes stored as `.parquet` files accomanied with a `data_attributes.csv` file into our `PROCESSED_KISPI_DATA_FOLDER`.  


### Convert .edf to parquet files
In order to better handle the dataset it would be advantagous to create a dataframe using trusty `pandas`, however we first needed to convert our EEG files from .edf format into parquet files.

To accomplish this we used the `edf2parquet` a simple utility package to convert EDF/EDF+ files into Apache Parquet format while preserving the EDF file header information and signal headers metadata information then we will be able to read the files directly<sup>1</sup>. 

We specifically used the built-in `AdvancedEdfToParquetConverter` class to convert the files and utilize the `exclude_singals` argument to exclude the following list of signals ["EEG A1", "EEG A2", "EKG", "EOG", "EMG", "PHO", "NASE"].  

In addition we create a data_attributes.csv file with all the important attributes we unvail.

---


[1] [edf2parquet repository](https://github.com/NarayanSchuetz/edf2parquet/tree/main)

This is just a quick test of the converter uncomment `#test_converter.convert()` to produce output.

In [2]:
test_edf_file = str(RAW_KISPI_DATA_FOLDER / Path("SE008_120418U-A.edf"))
test_parquet_output_dir = RAW_KISPI_DATA_FOLDER

test_converter = AdvancedEdfToParquetConverter(edf_file_path=test_edf_file,  # path to the EDF file
                                          # excluded signals     
                                          exclude_signals=["Audio"],  
                                          # output directory path
                                          parquet_output_dir=test_parquet_output_dir,
                                          # grouping signals with same sampling frequency
                                          group_by_sampling_freq=True,  
                                          # add a pd.DatetimeIndex to the resulting parquet files
                                          datetime_index=True,
                                          # specifies timezone location and start_date of the EDF file
                                          local_timezone=(pytz.timezone("Europe/Zurich"), pytz.timezone("Europe/Zurich")),
                                          # compression codec
                                          compression_codec="GZIP" 
                                          )

#test_converter.convert()

### Correct .edf header 'subject code' and 'admincode' for all annotated data files
Reason for this is that it causes issues when creating the data_attributes.csv file

- [x] This was completed manually using `EDFBrowser`'s Tools -> "Header editor repair" 

## Extract
We extract the files we want from our `RAW_KISPI_DATA_FOLDER` and save them into a list `filenames`

We will keep to list `filenames_a1` for our annotations from annotator1 and `filenames_a2` for our annotations from annotator2.

That way we avoid cross contamination of ground truth labels which could potentially ruin our end inter-rater agreement analysis.

We only make the list `filenames` containing all files to produce our `data_attributes.csv` file later.

In [3]:
# define the input and output folders
input_folder = RAW_KISPI_DATA_FOLDER
output_folder = PROCESSED_KISPI_DATA_FOLDER

# create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# get a list of all .edf files in the input folder
filenames_a1 = [filename for filename in os.listdir(input_folder) if filename.__contains__("annotated1") and filename.endswith('.edf')]
filenames_a2 = [filename for filename in os.listdir(input_folder) if filename.__contains__("annotated2") and filename.endswith('.edf')]
filenames = filenames_a1 + filenames_a2

#### TO DO: Add Andreas files and annotations
- [ ] add SE008_annotated2
- [ ] add SE008_annotated2_annotations.txt
- [ ] add SE026_annotated2
- [ ] add SE026_annotated2_annotations.txt  

In [4]:
print(filenames)
print(filenames_a1)
print(filenames_a2)

['SE008_annotated1.edf', 'SE021_annotated1.edf', 'SE026_annotated1.edf', 'SE038_annotated1.edf', 'SE021_annotated2.edf', 'SE038_annotated2.edf']
['SE008_annotated1.edf', 'SE021_annotated1.edf', 'SE026_annotated1.edf', 'SE038_annotated1.edf']
['SE021_annotated2.edf', 'SE038_annotated2.edf']


## Transform

### Convert Annotator-1 files

#### TO DO: Correct filename output
currently outputs: 'SE021 a1_2015-09-01_256.0'

goal: 'SE021_a1'

In [5]:
# convert all .edf files
for filename in tqdm(filenames_a1, desc='Converting EDF to Parquet'):
    # instantiate the AdvancedEdfToParquetConverter
    converter = AdvancedEdfToParquetConverter(
        edf_file_path=os.path.join(input_folder, filename),
        parquet_output_dir=output_folder,
        exclude_signals=["EEG A1", "EEG A2", "EKG", "EOG", "EMG", "PHO", "NASE"],
        group_by_sampling_freq=True,
        datetime_index=True,
        local_timezone=(pytz.timezone("Europe/Zurich"), pytz.timezone("Europe/Zurich")),
        compression_codec="GZIP"
    )
    
    # convert the EDF file to Parquet and save it to the output folder
    #converter.convert()

Converting EDF to Parquet:   0%|          | 0/4 [00:00<?, ?it/s]

Converting EDF to Parquet: 100%|██████████| 4/4 [00:00<00:00,  5.28it/s]


#### Read Converted Parquet Annotator-1 Files 
using handy pandas.

In [6]:
SE008_a1_parquet_file_path = PROCESSED_KISPI_DATA_FOLDER / Path("SE008_a1.parquet")
SE021_a1_parquet_file_path = PROCESSED_KISPI_DATA_FOLDER / Path("SE021_a1.parquet")
SE026_a1_parquet_file_path = PROCESSED_KISPI_DATA_FOLDER / Path("SE026_a1.parquet")
SE038_a1_parquet_file_path = PROCESSED_KISPI_DATA_FOLDER / Path("SE038_a1.parquet")

# convert WindowsPath to strings (to avoid encoding errors)
SE008_a1_parquet_file_path = str(SE008_a1_parquet_file_path)
SE021_a1_parquet_file_path = str(SE021_a1_parquet_file_path)
SE026_a1_parquet_file_path = str(SE026_a1_parquet_file_path)
SE038_a1_parquet_file_path = str(SE038_a1_parquet_file_path)

a1_parquet_file_paths = [SE008_a1_parquet_file_path, SE021_a1_parquet_file_path, SE026_a1_parquet_file_path, SE038_a1_parquet_file_path]

#### Read Converted Parqeut Annotator-1 File 
using the `ParquetReader` directly tranfrom into pandas_df

In [7]:
from edf2parquet.readers import ParquetReader

read_SE021_a1 = ParquetReader(parquet_file_path=SE021_a1_parquet_file_path)
SE021_a1_df = pd.read_parquet(SE021_a1_parquet_file_path, engine='pyarrow')

Read one file and get file header

In [8]:
read_SE021_a1.get_file_header()

{'technician': '',
 'recording_additional': '',
 'patientname': 'X',
 'patient_additional': '',
 'patientcode': 'SE021 a1',
 'equipment': 'Deltamed',
 'admincode': 'SE021 150901U-C 0000',
 'sex': '',
 'startdate': Timestamp('2015-09-01 16:06:27+0200', tz='Europe/Zurich'),
 'birthdate': '',
 'gender': '',
 'tz_recording': 'Europe/Zurich',
 'tz_startdatetime': 'Europe/Zurich'}

Read rest of files, get header, transform into df's

In [9]:
read_SE008_a1 = ParquetReader(parquet_file_path=SE008_a1_parquet_file_path)
SE008_a1_df = pd.read_parquet(SE008_a1_parquet_file_path, engine='pyarrow')

read_SE026_a1 = ParquetReader(parquet_file_path=SE026_a1_parquet_file_path)
SE026_a1_df = pd.read_parquet(SE026_a1_parquet_file_path, engine='pyarrow')

read_SE038_a1 = ParquetReader(parquet_file_path=SE038_a1_parquet_file_path)
SE038_a1_df = pd.read_parquet(SE038_a1_parquet_file_path, engine='pyarrow')

In [10]:
print(read_SE008_a1.get_file_header())
print(read_SE026_a1.get_file_header())
print(read_SE038_a1.get_file_header())

{'technician': '', 'recording_additional': '', 'patientname': 'X', 'patient_additional': '', 'patientcode': 'SE008 a1', 'equipment': 'Deltamed', 'admincode': 'SE008 120418U-A', 'sex': '', 'startdate': Timestamp('2012-04-17 17:02:08+0200', tz='Europe/Zurich'), 'birthdate': '', 'gender': '', 'tz_recording': 'Europe/Zurich', 'tz_startdatetime': 'Europe/Zurich'}
{'technician': '', 'recording_additional': '', 'patientname': 'X', 'patient_additional': '', 'patientcode': 'SE026 a1', 'equipment': 'Deltamed', 'admincode': 'SE026 140520Q-A', 'sex': '', 'startdate': Timestamp('2014-05-20 08:30:52+0200', tz='Europe/Zurich'), 'birthdate': '', 'gender': '', 'tz_recording': 'Europe/Zurich', 'tz_startdatetime': 'Europe/Zurich'}
{'technician': '', 'recording_additional': '', 'patientname': 'X', 'patient_additional': '', 'patientcode': 'SE038 a1', 'equipment': 'Deltamed', 'admincode': 'SE038 180523D-D', 'sex': '', 'startdate': Timestamp('2018-05-23 11:05:57+0200', tz='Europe/Zurich'), 'birthdate': '', '

#### Filter Data
Based on annotation lengths

##### Filter SE008_a1
by its unique `cutoff_time_SE008`

In [11]:
print(SE008_a1_df.shape)

(15010560, 19)


In [12]:
SE008_a1_df = SE008_a1_df.copy()
SE008_a1_df.index = SE008_a1_df.index.tz_localize(None)

In [13]:
SE008_a1_df.head(-1)

,EEG Fp1,EEG Fp2,EEG F7,EEG F3,EEG Fz,EEG F4,EEG F8,EEG T3,EEG C3,EEG Cz,EEG C4,EEG T4,EEG T5,EEG P3,EEG Pz,EEG P4,EEG T6,EEG O1,EEG O2
2012-04-17 17:02:08.000000,107.303505,78.486382,163.326843,136.478592,-6.354085,123.591438,143.638138,102.649803,41.435799,84.392998,104.797668,-5.280156,121.980545,138.805450,73.116730,30.338522,-16.377432,63.988327,160.463028
2012-04-17 17:02:08.003906,107.303505,78.665367,164.400772,136.120621,-6.354085,121.622566,142.922180,101.217896,39.466927,81.350197,102.470818,-6.175097,119.653694,135.225677,70.073929,28.011673,-17.451363,61.482491,157.062256
2012-04-17 17:02:08.007812,108.914398,80.634239,167.980545,138.626465,-5.996109,123.412453,145.428009,103.723732,39.824902,81.887161,104.081711,-4.922179,118.937744,135.762650,71.147858,28.190662,-15.124514,59.513618,158.136185
2012-04-17 17:02:08.011718,108.914398,80.992218,167.443573,138.805450,-5.459144,123.591438,146.143967,102.470818,39.108948,82.424126,105.871597,-3.490272,118.400780,136.120621,73.116730,30.159533,-13.513618,58.439690,159.568100
2012-04-17 17:02:08.015625,108.556419,81.529182,166.727631,137.552536,-3.311284,124.486382,145.964981,101.575874,38.929962,84.392998,107.661476,-1.342412,120.190659,137.731522,75.622566,32.844357,-11.723736,60.229572,162.073929
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2012-04-18 09:19:22.976562,116.789886,45.552528,300.252930,105.334633,-1.521401,108.198441,-1210.766479,138.626465,104.439690,6.712062,107.482491,31.054474,-34.813229,28.011673,-100.143967,24.610895,57.902725,-674.159546,122.696495
2012-04-18 09:19:22.980468,115.715950,45.015564,298.105072,110.883270,46.089493,108.377434,-1048.424072,135.762650,101.038910,4.385214,108.198441,32.307392,-36.603111,26.579767,-102.291832,23.357977,56.649807,-674.338501,121.264595
2012-04-18 09:19:22.984375,117.147858,45.015564,299.357971,126.276268,110.167313,105.155640,-741.101196,131.108948,98.712059,5.996109,106.587547,31.054474,-36.424126,25.684824,-103.723732,21.389105,57.723736,-672.727600,118.758759
2012-04-18 09:19:22.988281,118.937744,47.521400,301.147858,122.338524,62.556419,105.692604,-670.221802,135.404663,98.712059,3.848249,103.544746,32.665371,-34.455254,24.252918,-104.439690,19.599222,57.902725,-674.159546,117.505836


In [14]:
cutoff_time_SE008 = pd.to_datetime("2012-04-18 03:21:21.000000")

In [15]:
SE008_a1_df = SE008_a1_df.loc[SE008_a1_df.index < cutoff_time_SE008]
print(SE008_a1_df.head(-1))


                               EEG Fp1    EEG Fp2      EEG F7      EEG F3  \
2012-04-17 17:02:08.000000  107.303505  78.486382  163.326843  136.478592   
2012-04-17 17:02:08.003906  107.303505  78.665367  164.400772  136.120621   
2012-04-17 17:02:08.007812  108.914398  80.634239  167.980545  138.626465   
2012-04-17 17:02:08.011718  108.914398  80.992218  167.443573  138.805450   
2012-04-17 17:02:08.015625  108.556419  81.529182  166.727631  137.552536   
...                                ...        ...         ...         ...   
2012-04-18 03:21:20.976562  149.186768  60.229572  223.466919  -24.968872   
2012-04-18 03:21:20.980468  141.311279  55.754864  214.517517   -3.490272   
2012-04-18 03:21:20.984375  128.961090  46.268482  201.630356   28.906614   
2012-04-18 03:21:20.988281  130.571991  47.700390  203.420227   28.548637   
2012-04-18 03:21:20.992187  143.459137  58.439690  216.844360   -3.669261   

                                EEG Fz      EEG F4      EEG F8      EEG T3 

`SE008_a1_df` went from [15010559 rows] to [9511167 rows]

##### Filter SE026_a1
by unique `cutoff_time_SE026`

In [16]:
SE026_a1_df = SE026_a1_df.copy()
SE026_a1_df.index = SE026_a1_df.index.tz_localize(None)

In [17]:
SE026_a1_df.head(-1)

,EEG Fp1,EEG Fp2,EEG F7,EEG F3,EEG Fz,EEG F4,EEG F8,EEG T3,EEG C3,EEG Cz,EEG C4,EEG T4,EEG T5,EEG P3,EEG Pz,EEG P4,EEG T6,EEG O1,EEG O2
2014-05-20 08:30:52.000000,-40.898834,92.089493,-12.260700,-146.143967,-72.937744,67.031128,166.190659,45.015564,-9.038911,69.536964,156.346298,130.750977,167.801559,206.284042,211.474701,176.214005,167.264587,162.968872,203.957199
2014-05-20 08:30:52.003906,-25.684824,102.649803,-12.439689,-136.657593,-37.319065,68.105057,165.832687,82.782104,-20.136187,86.898834,156.883270,130.035019,178.719849,205.926071,218.813232,201.809341,171.023346,167.085602,213.801559
2014-05-20 08:30:52.007812,-23.357977,103.365761,-9.038911,-129.319061,-34.813229,74.011673,168.875488,84.929962,-23.715954,82.245132,151.871597,132.361862,190.354080,207.178986,225.614792,210.221786,186.058365,178.182877,231.700394
2014-05-20 08:30:52.011718,-33.202335,95.669258,-4.206226,-132.182877,-68.642021,80.097275,177.108948,50.564201,-21.747082,58.976654,146.859924,140.595337,192.680939,206.821014,229.910507,196.618683,198.587555,193.396881,246.914398
2014-05-20 08:30:52.015625,-40.361866,92.984436,-1.879377,-139.163422,-91.552528,78.844360,183.731522,28.906614,-22.284046,44.836575,143.280151,141.490265,188.027237,202.525299,230.984436,180.867706,194.828796,198.945526,248.167313
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-05-20 09:40:12.976562,-224.003891,-63.630352,-142.206223,-335.513611,-249.241241,-18.704281,58.439690,-104.260704,-292.198456,-219.708176,47.879379,7.785992,-73.474709,-156.704285,-200.556427,-34.455254,-24.252918,-117.326851,-135.404663
2014-05-20 09:40:12.980468,-227.941635,-70.252922,-157.778214,-337.303497,-251.031128,-14.408561,56.112839,-105.513618,-288.618683,-234.385208,40.182880,7.785992,-64.167313,-152.229568,-188.206223,-32.128407,-21.568094,-96.564201,-122.338524
2014-05-20 09:40:12.984375,-227.404663,-63.988327,-159.926071,-334.976654,-248.883270,-11.186770,59.334629,-103.186768,-284.501953,-234.206223,40.182880,9.575875,-62.019455,-149.365753,-181.762650,-29.801556,-19.778210,-89.941635,-116.610893
2014-05-20 09:40:12.988281,-224.361862,-55.038910,-147.217896,-330.859924,-246.735413,-14.229572,63.988327,-97.996109,-280.922180,-222.571991,44.657589,5.817121,-63.451363,-146.322952,-186.774323,-34.276264,-24.431906,-100.143967,-120.727623


In [18]:
cutoff_time_SE026 = pd.to_datetime("2014-05-20 09:00:52.000000")

In [19]:
SE026_a1_df = SE026_a1_df.loc[SE026_a1_df.index < cutoff_time_SE026]
print(SE026_a1_df.head(-1))

                              EEG Fp1     EEG Fp2     EEG F7      EEG F3  \
2014-05-20 08:30:52.000000 -40.898834   92.089493 -12.260700 -146.143967   
2014-05-20 08:30:52.003906 -25.684824  102.649803 -12.439689 -136.657593   
2014-05-20 08:30:52.007812 -23.357977  103.365761  -9.038911 -129.319061   
2014-05-20 08:30:52.011718 -33.202335   95.669258  -4.206226 -132.182877   
2014-05-20 08:30:52.015625 -40.361866   92.984436  -1.879377 -139.163422   
...                               ...         ...        ...         ...   
2014-05-20 09:00:51.976562 -60.766537  -60.587547  10.112841   40.540855   
2014-05-20 09:00:51.980468 -68.642021  -66.852142   6.712062   36.066147   
2014-05-20 09:00:51.984375 -66.136185  -66.136185   7.249027   38.392998   
2014-05-20 09:00:51.988281 -69.000000  -68.463036   4.922179   38.392998   
2014-05-20 09:00:51.992187 -74.011673  -74.011673   2.237354   32.844357   

                               EEG Fz     EEG F4      EEG F8     EEG T3  \
2014-05-20 0

`SE026_a1_df` went from [1065215 rows] to [460799 rows]

##### Filter SE038_a1
by unique `cutoff_time_SE038`

In [20]:
SE038_a1_df = SE038_a1_df.copy()
SE038_a1_df.index = SE038_a1_df.index.tz_localize(None)

In [21]:
SE038_a1_df.head(-1)

,EEG Fp1,EEG Fp2,EEG F7,EEG F3,EEG Fz,EEG F4,EEG F8,EEG T3,EEG C3,EEG Cz,EEG C4,EEG T4,EEG T5,EEG P3,EEG Pz,EEG P4,EEG T6,EEG O1,EEG O2
2018-05-23 11:05:57.000000,153.840469,-192.322952,-228.836578,49.311283,-100.680931,-157.599228,-234.743195,-129.856033,171.560318,-46.626461,-227.404663,-159.210114,-294.346313,-307.770416,-6.533074,-132.003891,-59.334629,-92.089493,28.548637
2018-05-23 11:05:57.003906,154.019455,-189.280151,-220.961090,45.910507,-101.933853,-163.326843,-230.626465,-135.046692,78.486382,-45.015564,-227.404663,-158.852142,-299.000000,-311.171204,-0.805447,-127.887161,-59.334629,-92.447472,39.287937
2018-05-23 11:05:57.007812,185.163422,-161.715958,-207.715958,91.910507,-82.066147,-130.571991,-210.042801,-120.369652,189.101166,-29.622568,-215.054474,-136.657593,-279.311279,-291.840454,7.249027,-118.400780,-38.214008,-82.066147,11.186770
2018-05-23 11:05:57.011718,200.556427,-150.797668,-209.505844,123.949417,-70.789886,-107.303505,-204.852142,-115.357979,361.287933,-22.463036,-208.073929,-127.529182,-273.583649,-283.964966,4.385214,-118.400780,-31.054474,-84.750977,-23.536964
2018-05-23 11:05:57.015625,178.540863,-171.381317,-228.120621,94.237350,-85.287941,-125.202332,-224.719849,-137.731522,364.509735,-33.381325,-216.486374,-147.933853,-298.821014,-308.128418,-9.754864,-131.466919,-51.996109,-107.303505,-23.357977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-05-23 12:50:31.976562,22.284046,-95.848251,-13.334630,-219.887161,-55.754864,33.202335,-37.856030,26.758755,-100.859924,-64.883270,-124.486382,-36.066147,-409.077820,-317.256805,-186.774323,-180.330734,-136.836578,-655.544739,-295.062256
2018-05-23 12:50:31.980468,22.105059,-89.941635,6.712062,-218.455246,-57.902725,34.634243,-46.805447,41.614784,-96.385216,-62.377434,-125.381325,-10.470818,-406.571991,-291.124512,-172.992218,-166.011673,-132.719849,-656.976624,-274.120636
2018-05-23 12:50:31.984375,21.210117,-79.560310,29.622568,-215.233459,-60.050583,36.424126,-47.521400,91.910507,-82.961090,-54.859921,-127.529182,-21.389105,-396.190674,-239.038910,-152.766541,-139.342407,-127.350197,-656.976624,-238.680939
2018-05-23 12:50:31.988281,23.536964,-87.614784,34.097275,-210.042801,-58.976654,39.645916,-44.657589,90.478600,-78.486382,-51.280155,-125.739296,-26.221790,-388.852142,-229.015564,-150.081711,-133.256805,-121.980545,-649.459167,-229.194550


In [22]:
cutoff_time_SE038 = pd.to_datetime("2018-05-23 11:36:27.000000")

In [23]:
SE038_a1_df = SE038_a1_df.loc[SE038_a1_df.index < cutoff_time_SE038]
print(SE038_a1_df.head(-1))

                               EEG Fp1     EEG Fp2      EEG F7      EEG F3  \
2018-05-23 11:05:57.000000  153.840469 -192.322952 -228.836578   49.311283   
2018-05-23 11:05:57.003906  154.019455 -189.280151 -220.961090   45.910507   
2018-05-23 11:05:57.007812  185.163422 -161.715958 -207.715958   91.910507   
2018-05-23 11:05:57.011718  200.556427 -150.797668 -209.505844  123.949417   
2018-05-23 11:05:57.015625  178.540863 -171.381317 -228.120621   94.237350   
...                                ...         ...         ...         ...   
2018-05-23 11:36:26.976562   46.984436  -35.350193  -50.564201   13.334630   
2018-05-23 11:36:26.980468   40.361866  -33.202335  -24.610895    9.217898   
2018-05-23 11:36:26.984375   34.276264  -31.591440   10.112841    7.964981   
2018-05-23 11:36:26.988281   38.750973  -30.338522    8.322957    9.933852   
2018-05-23 11:36:26.992187   48.237354  -29.443581  -26.579767   13.155642   

                                EEG Fz      EEG F4      EEG F8 

`SE038_a1_df` went from [1606399 rows] to [468479 rows]

#### Create Bipolar Channel Pairs

In [24]:
# create bipolar channels from the original channels
def create_bipolar_channels(df):
  """
  Creates bipolar channels from the original channels.

  Args:
    df: A Pandas DataFrame with the EEG data, where the index is a datetime object.

  Returns:
    A DataFrame with bipolar channels.
  """

  # create list of bipolar pairs
  bipolar_pairs = [
      ('Fp1-F7', 'EEG Fp1', 'EEG F7'),
      ('F7-T3', 'EEG F7', 'EEG T3'),
      ('T3-T5', 'EEG T3', 'EEG T5'),
      ('Fp1-F3', 'EEG Fp1', 'EEG F3'),
      ('F3-C3', 'EEG F3', 'EEG C3'),
      ('C3-P3', 'EEG C3', 'EEG P3'),
      ('P3-O1', 'EEG P3', 'EEG O1'),
      ('Fz-Cz', 'EEG Fz', 'EEG Cz'),
      ('Cz-Pz', 'EEG Cz', 'EEG Pz'),
      ('Fp2-F4', 'EEG Fp2', 'EEG F4'),
      ('F4-C4', 'EEG F4', 'EEG C4'),
      ('C4-P4', 'EEG C4', 'EEG P4'),
      ('P4-O2', 'EEG P4', 'EEG O2'),
      ('Fp2-F8', 'EEG Fp2', 'EEG F8'),
      ('F8-T4', 'EEG F8', 'EEG T4'),
      ('T4-T6', 'EEG T4', 'EEG T6'),
      ('T6-O2', 'EEG T6', 'EEG O2'),
  ]

  # create a new empty DataFrame
  bipolar_df = pd.DataFrame(index=df.index)

  for pair in bipolar_pairs:
      # strip any leading/trailing spaces from column names
      ch1 = pair[1].strip()  
      ch2 = pair[2].strip()

      if ch1 in df.columns and ch2 in df.columns:
          bipolar_df[pair[0]] = df[ch1] - df[ch2]
      else:
          print(f"Warning: Channel(s) missing for pair {pair[0]}")

  return bipolar_df

In [25]:
SE008_a1_df = create_bipolar_channels(SE008_a1_df)
SE021_a1_df = create_bipolar_channels(SE021_a1_df)
SE026_a1_df = create_bipolar_channels(SE026_a1_df)
SE038_a1_df = create_bipolar_channels(SE038_a1_df)

In [26]:
print(f"SE008_a1_df",SE008_a1_df.head())
print(f"SE021_a1_df",SE021_a1_df.head())
print(f"SE026_a1_df",SE026_a1_df.head())
print(f"SE038_a1_df",SE038_a1_df.head())

SE008_a1_df                                Fp1-F7      F7-T3      T3-T5     Fp1-F3  \
2012-04-17 17:02:08.000000 -56.023338  60.677040 -19.330742 -29.175087   
2012-04-17 17:02:08.003906 -57.097267  63.182877 -18.435799 -28.817116   
2012-04-17 17:02:08.007812 -59.066147  64.256813 -15.214012 -29.712067   
2012-04-17 17:02:08.011718 -58.529175  64.972755 -15.929962 -29.891052   
2012-04-17 17:02:08.015625 -58.171211  65.151756 -18.614784 -28.996117   

                                F3-C3      C3-P3      P3-O1      Fz-Cz  \
2012-04-17 17:02:08.000000  95.042793 -97.369652  74.817123 -90.747086   
2012-04-17 17:02:08.003906  96.653694 -95.758751  73.743187 -87.704285   
2012-04-17 17:02:08.007812  98.801559 -95.937744  76.249031 -87.883270   
2012-04-17 17:02:08.011718  99.696503 -97.011673  77.680931 -87.883270   
2012-04-17 17:02:08.015625  98.622574 -98.801559  77.501953 -87.704285   

                                Cz-Pz     Fp2-F4      F4-C4      C4-P4  \
2012-04-17 17:02:08.0000

In [27]:
print(SE008_a1_df.shape)
print(SE021_a1_df.shape)
print(SE026_a1_df.shape)
print(SE038_a1_df.shape)

(9511168, 17)
(368640, 17)
(460800, 17)
(468480, 17)


In [28]:
print(SE008_a1_df.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9511168 entries, 2012-04-17 17:02:08 to 2012-04-18 03:21:20.996093
Data columns (total 17 columns):
 #   Column  Dtype  
---  ------  -----  
 0   Fp1-F7  float32
 1   F7-T3   float32
 2   T3-T5   float32
 3   Fp1-F3  float32
 4   F3-C3   float32
 5   C3-P3   float32
 6   P3-O1   float32
 7   Fz-Cz   float32
 8   Cz-Pz   float32
 9   Fp2-F4  float32
 10  F4-C4   float32
 11  C4-P4   float32
 12  P4-O2   float32
 13  Fp2-F8  float32
 14  F8-T4   float32
 15  T4-T6   float32
 16  T6-O2   float32
dtypes: float32(17)
memory usage: 689.4 MB
None


#### Create Ground Truth Labels
Using `_annotations.txt` files for all patients

In [29]:
SE008_a1_annotations = pd.read_csv(RAW_KISPI_DATA_FOLDER / "SE008_annotated1_annotations.txt")
SE008_a1_annotations.head(-1)

,Onset,Duration,Annotation
0,2012-04-17T17:02:12.5802030,NaN,Burst starts
1,2012-04-17T17:02:13.2416306,NaN,Burst end
2,2012-04-17T17:02:55.9220000,NaN,Burst starts
3,2012-04-17T17:02:57.5790000,NaN,Burst end
4,2012-04-17T17:03:00.5140000,NaN,Burst starts
...,...,...,...
286,2012-04-18T03:21:04.9590000,NaN,Burst starts
287,2012-04-18T03:21:05.9080000,NaN,Burst end
288,2012-04-18T03:21:08.9550000,NaN,Burst starts
289,2012-04-18T03:21:09.7740000,NaN,Burst end


In [30]:
SE008_a1_annotations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 292 entries, 0 to 291
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Onset       292 non-null    object 
 1   Duration    0 non-null      float64
 2   Annotation  292 non-null    object 
dtypes: float64(1), object(2)
memory usage: 7.0+ KB


In [31]:
SE021_a1_annotations = pd.read_csv(RAW_KISPI_DATA_FOLDER / "SE021_annotated1_annotations.txt")
SE026_a1_annotations = pd.read_csv(RAW_KISPI_DATA_FOLDER / "SE026_annotated1_annotations.txt")
SE038_a1_annotations = pd.read_csv(RAW_KISPI_DATA_FOLDER / "SE038_annotated1_annotations.txt")

##### SE008_a1 Ground Truth Labels
Create `SE008_a1_ground_truth_df`

In [32]:
SE008_a1_df.head(-1)

,Fp1-F7,F7-T3,T3-T5,Fp1-F3,F3-C3,C3-P3,P3-O1,Fz-Cz,Cz-Pz,Fp2-F4,F4-C4,C4-P4,P4-O2,Fp2-F8,F8-T4,T4-T6,T6-O2
2012-04-17 17:02:08.000000,-56.023338,60.677040,-19.330742,-29.175087,95.042793,-97.369652,74.817123,-90.747086,11.276268,-45.105057,18.793770,74.459145,-130.124512,-65.151756,148.918289,11.097277,-176.840454
2012-04-17 17:02:08.003906,-57.097267,63.182877,-18.435799,-28.817116,96.653694,-95.758751,73.743187,-87.704285,11.276268,-42.957199,19.151749,74.459145,-129.050583,-64.256813,149.097275,11.276265,-174.513611
2012-04-17 17:02:08.007812,-59.066147,64.256813,-15.214012,-29.712067,98.801559,-95.937744,76.249031,-87.883270,10.739304,-42.778214,19.330742,75.891052,-129.945526,-64.793770,150.350189,10.202334,-173.260696
2012-04-17 17:02:08.011718,-58.529175,64.972755,-15.929962,-29.891052,99.696503,-97.011673,77.680931,-87.883270,9.307396,-42.599220,17.719841,75.712067,-129.408569,-65.151749,149.634232,10.023346,-173.081726
2012-04-17 17:02:08.015625,-58.171211,65.151756,-18.614784,-28.996117,98.622574,-98.801559,77.501953,-87.704285,8.770432,-42.957199,16.824905,74.817123,-129.229568,-64.435799,147.307388,10.381323,-173.797668
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2012-04-18 03:21:20.976562,-74.280151,113.120613,-60.856026,174.155640,-65.867706,-104.887161,89.494171,-383.035034,-82.871597,-45.642025,-37.050583,57.276268,-174.871613,0.000000,64.077820,-138.715958,-125.649811
2012-04-18 03:21:20.980468,-73.206238,107.929970,-61.750969,144.801544,-42.241245,-99.875488,77.143974,-35.439686,-78.933853,-39.377430,-39.198441,56.202332,-177.377441,0.536964,61.214008,-135.494156,-126.007797
2012-04-18 03:21:20.984375,-72.669266,103.813232,-59.961090,100.054474,-3.758757,-92.536957,60.498055,384.645905,-72.848251,-35.797665,-40.988327,56.739296,-177.735413,0.536964,53.696499,-125.828796,-126.186768
2012-04-18 03:21:20.988281,-72.848236,105.603104,-57.813232,102.023354,-2.147861,-92.000000,62.645912,314.482483,-71.058365,-37.587551,-36.692604,60.319065,-177.735397,2.147861,54.591438,-123.143967,-125.291824


In [33]:
SE008_a1_annotations["Onset"] = pd.to_datetime(SE008_a1_annotations["Onset"], format="%Y-%m-%dT%H:%M:%S.%f")

SE008_a1_timestamps = SE008_a1_df.index
SE008_a1_ground_truth_df = pd.DataFrame(
    {"timestamp" : SE008_a1_timestamps, "ground_truth" : 0}
    )

SE008_a1_ground_truth_df["timestamp"] = pd.to_datetime(SE008_a1_ground_truth_df["timestamp"], format="%Y-%m-%dT%H:%M:%S.%f")

for index, row in SE008_a1_annotations.iterrows():
    if row["Annotation"] == "Burst starts":
        start_time = row["Onset"]
        try:
            # find the next row where the annotation is "Burst end"
            end_time = SE008_a1_annotations.loc[
                (SE008_a1_annotations["Annotation"] == "Burst end") &
                (SE008_a1_annotations.index > index), "Onset"
            ].iloc[0]
        
            SE008_a1_ground_truth_df.loc[
                (SE008_a1_ground_truth_df["timestamp"] >= start_time) 
                & (SE008_a1_ground_truth_df["timestamp"] <= end_time), 
                "ground_truth",] = 1
        except IndexError:
            print(f"Warning: No matching 'Burst end' annotation found for 'Burst starts' at {start_time}")

In [34]:
SE008_a1_ground_truth_df.head()

,timestamp,ground_truth
0,2012-04-17 17:02:08.000000,0
1,2012-04-17 17:02:08.003906,0
2,2012-04-17 17:02:08.007812,0
3,2012-04-17 17:02:08.011718,0
4,2012-04-17 17:02:08.015625,0


In [35]:
print(SE008_a1_ground_truth_df.shape)
print(SE008_a1_df.shape)

(9511168, 2)
(9511168, 17)


In [36]:
SE008_a1_ground_truth_df.to_csv(
    RAW_KISPI_DATA_FOLDER / "SE008_a1_ground_truth.csv",
    date_format="%Y-%m-%d %H:%M:%S.%f"
)

##### SE021_a1 Ground Truth Labels
Create `SE021_a1_ground_truth_df`

In [37]:
SE021_a1_annotations.head(-1)

,Onset,Duration,Annotation
0,2015-09-01T16:06:28.4270000,NaN,Burst start
1,2015-09-01T16:06:31.4130000,NaN,Burst end
2,2015-09-01T16:06:53.0490000,NaN,Burst start
3,2015-09-01T16:06:56.1850000,NaN,Burst end
4,2015-09-01T16:07:03.7870000,NaN,Burst start
...,...,...,...
100,2015-09-01T16:27:11.5340000,NaN,Burst start
101,2015-09-01T16:27:14.6310000,NaN,Burst end
102,2015-09-01T16:29:50.5230000,NaN,Burst start
103,2015-09-01T16:29:53.6010000,NaN,Burst end


In [38]:
SE021_a1_df.index = SE021_a1_df.index.tz_localize(None)

In [39]:
SE021_a1_annotations["Onset"] = pd.to_datetime(SE021_a1_annotations["Onset"], format="%Y-%m-%dT%H:%M:%S.%f")

SE021_a1_timestamps = SE021_a1_df.index

SE021_a1_ground_truth_df = pd.DataFrame(
    {"timestamp" : SE021_a1_timestamps, "ground_truth" : 0}
)

SE021_a1_ground_truth_df["timestamp"] = pd.to_datetime(SE021_a1_ground_truth_df["timestamp"], format="%Y-%m-%dT%H:%M:%S.%f")

for index, row in SE021_a1_annotations.iterrows():
    if row["Annotation"] == "Burst start":
        start_time = row["Onset"]
        try:
            # find the next row where the annotation is "Burst end"
            end_time = SE021_a1_annotations.loc[
                (SE021_a1_annotations["Annotation"] == "Burst end") &
                (SE021_a1_annotations.index > index), "Onset"
            ].iloc[0]

            print(f"Labeling from {start_time} to {end_time}")  # Print the interval being labeled

            SE021_a1_ground_truth_df.loc[
                (SE021_a1_ground_truth_df["timestamp"] >= start_time)
                & (SE021_a1_ground_truth_df["timestamp"] <= end_time),
                "ground_truth",
            ] = 1
        except IndexError:
            print(f"Warning: No matching 'Burst end' annotation found for 'Burst start' at {start_time}")

Labeling from 2015-09-01 16:06:28.427000 to 2015-09-01 16:06:31.413000
Labeling from 2015-09-01 16:06:53.049000 to 2015-09-01 16:06:56.185000
Labeling from 2015-09-01 16:07:03.787000 to 2015-09-01 16:07:06.472000
Labeling from 2015-09-01 16:07:08.145000 to 2015-09-01 16:07:11.616000
Labeling from 2015-09-01 16:07:58.715000 to 2015-09-01 16:08:01.701000
Labeling from 2015-09-01 16:08:18.518000 to 2015-09-01 16:08:22.153000
Labeling from 2015-09-01 16:08:37.584000 to 2015-09-01 16:08:40.982000
Labeling from 2015-09-01 16:08:54.114000 to 2015-09-01 16:08:57.258000
Labeling from 2015-09-01 16:09:42.254000 to 2015-09-01 16:09:45.424000
Labeling from 2015-09-01 16:09:48.559000 to 2015-09-01 16:09:52.454000
Labeling from 2015-09-01 16:10:07.675000 to 2015-09-01 16:10:11.618000
Labeling from 2015-09-01 16:10:19.981000 to 2015-09-01 16:10:23.275000
Labeling from 2015-09-01 16:10:42.228000 to 2015-09-01 16:10:45.705000
Labeling from 2015-09-01 16:11:11.628000 to 2015-09-01 16:11:14.994000
Labeli

In [40]:
SE021_a1_ground_truth_df.head(-1)

,timestamp,ground_truth
0,2015-09-01 16:06:27.000000,0
1,2015-09-01 16:06:27.003906,0
2,2015-09-01 16:06:27.007812,0
3,2015-09-01 16:06:27.011718,0
4,2015-09-01 16:06:27.015625,0
...,...,...
368634,2015-09-01 16:30:26.976562,0
368635,2015-09-01 16:30:26.980468,0
368636,2015-09-01 16:30:26.984375,0
368637,2015-09-01 16:30:26.988281,0


In [41]:
print(SE021_a1_ground_truth_df.shape)
print(SE021_a1_df.shape)

(368640, 2)
(368640, 17)


In [42]:
SE021_a1_ground_truth_df.to_csv(
    RAW_KISPI_DATA_FOLDER / "SE021_a1_ground_truth.csv",
    date_format="%Y-%m-%d %H:%M:%S.%f")

##### SE026_a1 Ground Truth Labels
Create `SE026_a1_ground_truth_df`

In [43]:
SE026_a1_annotations

,Onset,Duration,Annotation
0,2014-05-20T08:30:58.0110000,NaN,burst
1,2014-05-20T08:31:00.6570000,NaN,burst
2,2014-05-20T08:32:33.7090000,NaN,burst
3,2014-05-20T08:32:34.7430000,NaN,burst
4,2014-05-20T08:33:49.7850000,NaN,burst
...,...,...,...
131,2014-05-20T08:53:41.0400000,NaN,burst
132,2014-05-20T08:54:40.9620000,NaN,burst
133,2014-05-20T08:54:41.9700000,NaN,burst
134,2014-05-20T08:55:11.9350000,NaN,burst


In [44]:
SE026_a1_annotations["Onset"] = pd.to_datetime(SE026_a1_annotations["Onset"], format="%Y-%m-%dT%H:%M:%S.%f")

SE026_a1_timestamps = SE026_a1_df.index

SE026_a1_ground_truth_df = pd.DataFrame(
    {"timestamp" : SE026_a1_timestamps, "ground_truth" : 0}
    )

SE026_a1_ground_truth_df["timestamp"] = pd.to_datetime(SE026_a1_ground_truth_df["timestamp"], format="%Y-%m-%dT%H:%M:%S.%f")

burst_count = 0
for index, row in SE026_a1_annotations.iterrows():
    if row["Annotation"] == "burst":
        burst_count += 1
        if burst_count % 2 == 0:  # Check if it's an even count (should be a closing 'burst')
            end_time = row["Onset"]

            print(f"Labeling from {start_time} to {end_time}")
            
            SE026_a1_ground_truth_df.loc[
                (SE026_a1_ground_truth_df["timestamp"] >= start_time)
                & (SE026_a1_ground_truth_df["timestamp"] <= end_time),
                "ground_truth",] = 1
        else:
            start_time = row["Onset"]

if burst_count % 2 != 0:
    print(f"Warning: Unclosed 'burst' at {start_time}")

Labeling from 2014-05-20 08:30:58.011000 to 2014-05-20 08:31:00.657000
Labeling from 2014-05-20 08:32:33.709000 to 2014-05-20 08:32:34.743000
Labeling from 2014-05-20 08:33:49.785000 to 2014-05-20 08:33:50.518000
Labeling from 2014-05-20 08:35:14.392000 to 2014-05-20 08:35:15.394000
Labeling from 2014-05-20 08:35:29.195000 to 2014-05-20 08:35:30.060000
Labeling from 2014-05-20 08:35:40.261000 to 2014-05-20 08:35:41.315000
Labeling from 2014-05-20 08:36:01.547000 to 2014-05-20 08:36:02.359000
Labeling from 2014-05-20 08:36:07.975000 to 2014-05-20 08:36:08.741000
Labeling from 2014-05-20 08:36:15.407000 to 2014-05-20 08:36:16.173000
Labeling from 2014-05-20 08:36:20.749000 to 2014-05-20 08:36:21.535000
Labeling from 2014-05-20 08:36:23.599000 to 2014-05-20 08:36:24.588000
Labeling from 2014-05-20 08:36:35.538000 to 2014-05-20 08:36:36.756000
Labeling from 2014-05-20 08:36:44.038000 to 2014-05-20 08:36:45.119000
Labeling from 2014-05-20 08:36:58.444000 to 2014-05-20 08:37:00.049000
Labeli

In [45]:
SE026_a1_ground_truth_df.head()

,timestamp,ground_truth
0,2014-05-20 08:30:52.000000,0
1,2014-05-20 08:30:52.003906,0
2,2014-05-20 08:30:52.007812,0
3,2014-05-20 08:30:52.011718,0
4,2014-05-20 08:30:52.015625,0


In [46]:
print(SE026_a1_ground_truth_df.shape)
print(SE026_a1_df.shape)

(460800, 2)
(460800, 17)


In [47]:
SE026_a1_ground_truth_df.to_csv(
    RAW_KISPI_DATA_FOLDER / "SE026_a1_ground_truth.csv",
    date_format="%Y-%m-%d %H:%M:%S.%f")

##### SE038_a1 Ground Truth Labels
Create `SE038_a1_ground_truth_df`

In [48]:
SE038_a1_annotations.head(-1)

,Onset,Duration,Annotation
0,2018-05-23T11:06:21.1931892,NaN,burst starts
1,2018-05-23T11:06:21.7105435,NaN,burst end
2,2018-05-23T11:06:27.8244924,NaN,burst starts
3,2018-05-23T11:06:28.3491617,NaN,burst end
4,2018-05-23T11:06:31.0341650,NaN,burst starts
...,...,...,...
465,2018-05-23T11:36:14.6200000,NaN,burst
466,2018-05-23T11:36:15.4190000,NaN,burst
467,2018-05-23T11:36:18.5560000,NaN,burst
468,2018-05-23T11:36:19.1320000,NaN,burst


In [49]:
SE038_a1_annotations.iloc[158:182]

,Onset,Duration,Annotation
158,2018-05-23T11:13:38.8137819,NaN,burst starts
159,2018-05-23T11:13:39.1010000,NaN,burst end
160,2018-05-23T11:13:43.0060000,0.858,burst
161,2018-05-23T11:13:47.7210000,0.590,burst
162,2018-05-23T11:17:48.8244950,0.590,burst
163,2018-05-23T11:17:54.6860000,0.551,burst
164,2018-05-23T11:17:55.2420000,0.551,burst
165,2018-05-23T11:17:57.3640000,0.551,burst
166,2018-05-23T11:18:01.8100000,0.551,burst
167,2018-05-23T11:18:02.3600000,0.551,burst


In [50]:
SE038_a1_annotations["Onset"] = pd.to_datetime(SE038_a1_annotations["Onset"], format="%Y-%m-%dT%H:%M:%S.%f")

SE038_a1_timestamps = SE038_a1_df.index

SE038_a1_ground_truth_df = pd.DataFrame(
    {"timestamp" : SE038_a1_timestamps, "ground_truth" : 0}
)

SE038_a1_ground_truth_df["timestamp"] = pd.to_datetime(SE038_a1_ground_truth_df["timestamp"], format="%Y-%m-%dT%H:%M:%S.%f")

# Handling "burst-starts" and "burst-end" pairs
start_time = None
burst_start_count = 0
burst_end_count = 0
for index, row in SE038_a1_annotations.iterrows():
    if row["Annotation"] == "burst starts":
        burst_start_count += 1
        if start_time is None:
            start_time = row["Onset"]
        else:
            print(f"Warning: Unexpected 'burst starts' at {row['Onset']} without a matching 'burst end'")
    elif row["Annotation"] == "burst end":
        burst_end_count += 1
        if start_time is not None:
            end_time = row["Onset"]

            print(f"Labeling from {start_time} to {end_time}")

            SE038_a1_ground_truth_df.loc[
                (SE038_a1_ground_truth_df["timestamp"] >= start_time)
                & (SE038_a1_ground_truth_df["timestamp"] <= end_time),
                "ground_truth",
            ] = 1
            start_time = None  # Reset start_time for the next pair
        else:
            print(f"Warning: Unexpected 'burst end' at {row['Onset']} without a matching 'burst starts'")

if start_time is not None:
    print(f"Warning: Unclosed 'burst starts' at {start_time}")

if burst_start_count == burst_end_count:
    print("'burst-starts' and 'burst-end' pairs are even")
else:
    print(f"Warning: Uneven number of 'burst-starts' ({burst_start_count}) and 'burst-end' ({burst_end_count}) annotations")

# Handling simple "burst" with duration
for index, row in SE038_a1_annotations.iterrows():
    if row["Annotation"] == "burst" and pd.notna(row["Duration"]):
        start_time = row["Onset"]
        end_time = start_time + pd.to_timedelta(row["Duration"], unit='s')  # Calculate end_time using duration
        
        print(f"Labeling from {start_time} to {end_time}")  # Optional: Print the interval being labeled

        SE038_a1_ground_truth_df.loc[
            (SE038_a1_ground_truth_df["timestamp"] >= start_time)
            & (SE038_a1_ground_truth_df["timestamp"] <= end_time),
            "ground_truth",
        ] = 1

Labeling from 2018-05-23 11:06:21.193189200 to 2018-05-23 11:06:21.710543500
Labeling from 2018-05-23 11:06:27.824492400 to 2018-05-23 11:06:28.349161700
Labeling from 2018-05-23 11:06:31.034165 to 2018-05-23 11:06:31.649751100
Labeling from 2018-05-23 11:06:38.055432200 to 2018-05-23 11:06:38.635998600
Labeling from 2018-05-23 11:06:42.714895100 to 2018-05-23 11:06:43.124194400
Labeling from 2018-05-23 11:06:49.029000 to 2018-05-23 11:06:49.370000
Labeling from 2018-05-23 11:06:53.322000 to 2018-05-23 11:06:53.761000
Labeling from 2018-05-23 11:06:58.492000 to 2018-05-23 11:06:58.898000
Labeling from 2018-05-23 11:07:02.805000 to 2018-05-23 11:07:03.296000
Labeling from 2018-05-23 11:07:06.726000 to 2018-05-23 11:07:07.270000
Labeling from 2018-05-23 11:07:10.401000 to 2018-05-23 11:07:10.918000
Labeling from 2018-05-23 11:07:15.754000 to 2018-05-23 11:07:16.219000
Labeling from 2018-05-23 11:07:21.711000 to 2018-05-23 11:07:22.314000
Labeling from 2018-05-23 11:07:27.556000 to 2018-0

In [51]:
SE038_a1_ground_truth_df.head()

,timestamp,ground_truth
0,2018-05-23 11:05:57.000000,0
1,2018-05-23 11:05:57.003906,0
2,2018-05-23 11:05:57.007812,0
3,2018-05-23 11:05:57.011718,0
4,2018-05-23 11:05:57.015625,0


In [52]:
print(SE038_a1_ground_truth_df.shape)
print(SE038_a1_df.shape)

(468480, 2)
(468480, 17)


In [53]:
SE038_a1_ground_truth_df.to_csv(
    RAW_KISPI_DATA_FOLDER / "SE038_a1_ground_truth.csv",
    date_format="%Y-%m-%d %H:%M:%S.%f")

### Merge Annotator-1 Dataframes

In [54]:
bipolar_SE008_a1 = SE008_a1_df.copy()    
bipolar_SE021_a1 = SE021_a1_df.copy()    
bipolar_SE026_a1 = SE026_a1_df.copy()    
bipolar_SE038_a1 = SE038_a1_df.copy()

In [55]:
bipolar_SE008_a1_merged = pd.merge(bipolar_SE008_a1, SE008_a1_ground_truth_df, left_index=True, right_on="timestamp") 
bipolar_SE021_a1_merged = pd.merge(bipolar_SE021_a1, SE021_a1_ground_truth_df, left_index=True, right_on="timestamp")
bipolar_SE026_a1_merged = pd.merge(bipolar_SE026_a1, SE026_a1_ground_truth_df, left_index=True, right_on="timestamp")
bipolar_SE038_a1_merged = pd.merge(bipolar_SE038_a1, SE038_a1_ground_truth_df, left_index=True, right_on="timestamp")

In [56]:
print(bipolar_SE008_a1_merged.shape)
print(bipolar_SE021_a1_merged.shape)
print(bipolar_SE026_a1_merged.shape)
print(bipolar_SE038_a1_merged.shape)

(9511168, 19)
(368640, 19)
(460800, 19)
(468480, 19)


#### Drop Datetime Index
We now drop the current datatime index.

This will allows us to create our `data_learning` (Train) and `data_testing` (Test) datasets.

In [57]:
bipolar_SE008_a1_merged.reset_index(drop=True, inplace=True)
print(bipolar_SE008_a1_merged.head(-1))

bipolar_SE021_a1_merged.reset_index(drop=True, inplace=True)

bipolar_SE026_a1_merged.reset_index(drop=True, inplace=True)

bipolar_SE038_a1_merged.reset_index(drop=True, inplace=True)

            Fp1-F7       F7-T3      T3-T5      Fp1-F3      F3-C3       C3-P3  \
0       -56.023338   60.677040 -19.330742  -29.175087  95.042793  -97.369652   
1       -57.097267   63.182877 -18.435799  -28.817116  96.653694  -95.758751   
2       -59.066147   64.256813 -15.214012  -29.712067  98.801559  -95.937744   
3       -58.529175   64.972755 -15.929962  -29.891052  99.696503  -97.011673   
4       -58.171211   65.151756 -18.614784  -28.996117  98.622574  -98.801559   
...            ...         ...        ...         ...        ...         ...   
9511162 -74.280151  113.120613 -60.856026  174.155640 -65.867706 -104.887161   
9511163 -73.206238  107.929970 -61.750969  144.801544 -42.241245  -99.875488   
9511164 -72.669266  103.813232 -59.961090  100.054474  -3.758757  -92.536957   
9511165 -72.848236  105.603104 -57.813232  102.023354  -2.147861  -92.000000   
9511166 -73.385223  111.330742 -58.887154  147.128403 -38.661480  -97.906616   

             P3-O1       Fz-Cz      Cz-

#### Drop `Timestamp` column

In [58]:
bipolar_SE008_a1_merged.drop(columns="timestamp", inplace=True)
print(bipolar_SE008_a1_merged.head(-1))

bipolar_SE021_a1_merged.drop(columns="timestamp", inplace=True)
print(bipolar_SE021_a1_merged.head(-1))

bipolar_SE026_a1_merged.drop(columns="timestamp", inplace=True) 
print(bipolar_SE026_a1_merged.head(-1)) 

bipolar_SE038_a1_merged.drop(columns="timestamp", inplace=True)
print(bipolar_SE038_a1_merged.head(-1))

            Fp1-F7       F7-T3      T3-T5      Fp1-F3      F3-C3       C3-P3  \
0       -56.023338   60.677040 -19.330742  -29.175087  95.042793  -97.369652   
1       -57.097267   63.182877 -18.435799  -28.817116  96.653694  -95.758751   
2       -59.066147   64.256813 -15.214012  -29.712067  98.801559  -95.937744   
3       -58.529175   64.972755 -15.929962  -29.891052  99.696503  -97.011673   
4       -58.171211   65.151756 -18.614784  -28.996117  98.622574  -98.801559   
...            ...         ...        ...         ...        ...         ...   
9511162 -74.280151  113.120613 -60.856026  174.155640 -65.867706 -104.887161   
9511163 -73.206238  107.929970 -61.750969  144.801544 -42.241245  -99.875488   
9511164 -72.669266  103.813232 -59.961090  100.054474  -3.758757  -92.536957   
9511165 -72.848236  105.603104 -57.813232  102.023354  -2.147861  -92.000000   
9511166 -73.385223  111.330742 -58.887154  147.128403 -38.661480  -97.906616   

             P3-O1       Fz-Cz      Cz-

### Convert Annotator-2 Files

In [59]:
# convert all .edf files
for filename in tqdm(filenames_a2, desc='Converting EDF to Parquet'):
    # instantiate the AdvancedEdfToParquetConverter
    converter = AdvancedEdfToParquetConverter(
        edf_file_path=os.path.join(input_folder, filename),
        parquet_output_dir=output_folder,
        exclude_signals=["EEG A1", "EEG A2", "EKG", "EOG", "EMG", "PHO", "NASE"],
        group_by_sampling_freq=True,
        datetime_index=True,
        local_timezone=(pytz.timezone("Europe/Zurich"), pytz.timezone("Europe/Zurich")),
        compression_codec="GZIP"
    )
    
    # convert the EDF file to Parquet and save it to the output folder
    #converter.convert()

Converting EDF to Parquet: 100%|██████████| 2/2 [00:00<00:00, 12.22it/s]


#### TO DO: Convert remaining (Andreas) annotated files
- [X] ~~Convert SE038_annotated2.edf~~
- [ ] Convert SE008_annotated2.edf
- [ ] Convert SE026_annotated2.edf

#### Read Converted Parquet Annotator-2 File 
using handy pandas.

In [60]:
#SE008_a2_parquet_file_path = PROCESSED_KISPI_DATA_FOLDER / Path("SE008_a2.parquet") FILE CURRENTLY DOESN'T EXIST
SE021_a2_parquet_file_path = PROCESSED_KISPI_DATA_FOLDER / Path("SE021_a2.parquet")
#SE026_a2_parquet_file_path = PROCESSED_KISPI_DATA_FOLDER / Path("SE026_a2.parquet") FILE CURRENTLY DOESN'T EXIST
SE038_a2_parquet_file_path = PROCESSED_KISPI_DATA_FOLDER / Path("SE038_a2.parquet")

# convert WindowsPath to strings (to avoid encoding errors)
#SE008_a2_parquet_file_pat = str(SE008_a2_parquet_file_pat)
SE021_a2_parquet_file_path = str(SE021_a2_parquet_file_path)
#SE026_a2_parquet_file_pat = str(SE026_a2_parquet_file_path)
SE038_a2_parquet_file_path = str(SE038_a2_parquet_file_path)

#### Read Converted Parqeut Annotator-2 File 
using the `ParquetReader` directly

In [61]:
from edf2parquet.readers import ParquetReader

read_SE021_a2 = ParquetReader(parquet_file_path=SE021_a2_parquet_file_path)
SE021_a2_df = pd.read_parquet(SE021_a2_parquet_file_path, engine='pyarrow')

In [62]:
read_SE021_a2.get_file_header()

{'technician': '',
 'recording_additional': '',
 'patientname': 'X',
 'patient_additional': '',
 'patientcode': 'SE021 a2',
 'equipment': 'Deltamed',
 'admincode': 'SE021 150901U-C 0000',
 'sex': '',
 'startdate': Timestamp('2015-09-01 16:06:27+0200', tz='Europe/Zurich'),
 'birthdate': '',
 'gender': '',
 'tz_recording': 'Europe/Zurich',
 'tz_startdatetime': 'Europe/Zurich'}

In [63]:
#read_SE008_a2 = ParquetReader(parquet_file_path=SE008_a2_parquet_file_path)
#SE008_a2_df = read_SE008_a2.get_pandas_dataframe(set_timezone=True)

#read_SE026_a2 = ParquetReader(parquet_file_path=SE026_a2_parquet_file_path)
#SE026_a2_df = read_SE026_a2.get_pandas_dataframe(set_timezone=True)

read_SE038_a2 = ParquetReader(parquet_file_path=SE038_a2_parquet_file_path)
SE038_a2_df = pd.read_parquet(SE038_a2_parquet_file_path, engine='pyarrow')  

In [64]:
#print(read_SE008_a2.get_file_header())
#print(read_SE026_a2.get_file_header())
print(read_SE038_a2.get_file_header())

{'technician': '', 'recording_additional': '', 'patientname': 'X', 'patient_additional': '', 'patientcode': 'SE038 a2', 'equipment': 'Deltamed', 'admincode': 'SE038 180523D-D', 'sex': '', 'startdate': Timestamp('2018-05-23 11:05:57+0200', tz='Europe/Zurich'), 'birthdate': '', 'gender': '', 'tz_recording': 'Europe/Zurich', 'tz_startdatetime': 'Europe/Zurich'}


#### Filter Data
Based on annotation lengths

##### Filter SE008_a2
by its unique `cutoff_time_SE008`

In [65]:
#print(SE008_a2_df.shape)

In [66]:
#SE008_a2_df = SE008_a2_df.copy()
#SE008_a2_df.index = SE008_a2_df.index.tz_localize(None)

In [67]:
#SE008_a2_df.head(-1)

`cutoff_time_SE008` previously defind

In [68]:

#SE008_a2_df = SE008_a2_df.loc[SE008_a2_df.index < cutoff_time_SE008]
#print(SE008_a1_df.head(-1))

`SE008_a2_df` went from [15010559 rows] to [9511167 rows]

##### Filter SE026_a2
by unique `cutoff_time_SE026`

In [69]:
#SE026_a2_df = SE026_a2_df.copy()
#SE026_a2_df.index = SE026_a2_df.index.tz_localize(None)

`cutoff_time_SE026` previously defined

In [70]:
#SE026_a2_df = SE026_a2_df.loc[SE026_a2_df.index < cutoff_time_SE026]
#print(SE026_a2_df.head(-1))

In [71]:
#SE026_a2_df.head(-1)

`SE026_a2_df` went from [1065215 rows] to [460799 rows]

##### Filter SE038_a2
by unique `cutoff_time_SE038`

In [72]:
SE038_a2_df = SE038_a2_df.copy()
SE038_a2_df.index = SE038_a2_df.index.tz_localize(None)

In [73]:
SE038_a2_df.head(-1)

,EEG Fp1,EEG Fp2,EEG F7,EEG F3,EEG Fz,EEG F4,EEG F8,EEG T3,EEG C3,EEG Cz,EEG C4,EEG T4,EEG T5,EEG P3,EEG Pz,EEG P4,EEG T6,EEG O1,EEG O2
2018-05-23 11:05:57.000000,153.840469,-192.322952,-228.836578,49.311283,-100.680931,-157.599228,-234.743195,-129.856033,171.560318,-46.626461,-227.404663,-159.210114,-294.346313,-307.770416,-6.533074,-132.003891,-59.334629,-92.089493,28.548637
2018-05-23 11:05:57.003906,154.019455,-189.280151,-220.961090,45.910507,-101.933853,-163.326843,-230.626465,-135.046692,78.486382,-45.015564,-227.404663,-158.852142,-299.000000,-311.171204,-0.805447,-127.887161,-59.334629,-92.447472,39.287937
2018-05-23 11:05:57.007812,185.163422,-161.715958,-207.715958,91.910507,-82.066147,-130.571991,-210.042801,-120.369652,189.101166,-29.622568,-215.054474,-136.657593,-279.311279,-291.840454,7.249027,-118.400780,-38.214008,-82.066147,11.186770
2018-05-23 11:05:57.011718,200.556427,-150.797668,-209.505844,123.949417,-70.789886,-107.303505,-204.852142,-115.357979,361.287933,-22.463036,-208.073929,-127.529182,-273.583649,-283.964966,4.385214,-118.400780,-31.054474,-84.750977,-23.536964
2018-05-23 11:05:57.015625,178.540863,-171.381317,-228.120621,94.237350,-85.287941,-125.202332,-224.719849,-137.731522,364.509735,-33.381325,-216.486374,-147.933853,-298.821014,-308.128418,-9.754864,-131.466919,-51.996109,-107.303505,-23.357977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-05-23 12:50:31.976562,22.284046,-95.848251,-13.334630,-219.887161,-55.754864,33.202335,-37.856030,26.758755,-100.859924,-64.883270,-124.486382,-36.066147,-409.077820,-317.256805,-186.774323,-180.330734,-136.836578,-655.544739,-295.062256
2018-05-23 12:50:31.980468,22.105059,-89.941635,6.712062,-218.455246,-57.902725,34.634243,-46.805447,41.614784,-96.385216,-62.377434,-125.381325,-10.470818,-406.571991,-291.124512,-172.992218,-166.011673,-132.719849,-656.976624,-274.120636
2018-05-23 12:50:31.984375,21.210117,-79.560310,29.622568,-215.233459,-60.050583,36.424126,-47.521400,91.910507,-82.961090,-54.859921,-127.529182,-21.389105,-396.190674,-239.038910,-152.766541,-139.342407,-127.350197,-656.976624,-238.680939
2018-05-23 12:50:31.988281,23.536964,-87.614784,34.097275,-210.042801,-58.976654,39.645916,-44.657589,90.478600,-78.486382,-51.280155,-125.739296,-26.221790,-388.852142,-229.015564,-150.081711,-133.256805,-121.980545,-649.459167,-229.194550


`cutoff_time_SE038` previously defined

In [74]:
SE038_a2_df = SE038_a2_df.loc[SE038_a2_df.index < cutoff_time_SE038]
print(SE038_a2_df.head(-1))

                               EEG Fp1     EEG Fp2      EEG F7      EEG F3  \
2018-05-23 11:05:57.000000  153.840469 -192.322952 -228.836578   49.311283   
2018-05-23 11:05:57.003906  154.019455 -189.280151 -220.961090   45.910507   
2018-05-23 11:05:57.007812  185.163422 -161.715958 -207.715958   91.910507   
2018-05-23 11:05:57.011718  200.556427 -150.797668 -209.505844  123.949417   
2018-05-23 11:05:57.015625  178.540863 -171.381317 -228.120621   94.237350   
...                                ...         ...         ...         ...   
2018-05-23 11:36:26.976562   46.984436  -35.350193  -50.564201   13.334630   
2018-05-23 11:36:26.980468   40.361866  -33.202335  -24.610895    9.217898   
2018-05-23 11:36:26.984375   34.276264  -31.591440   10.112841    7.964981   
2018-05-23 11:36:26.988281   38.750973  -30.338522    8.322957    9.933852   
2018-05-23 11:36:26.992187   48.237354  -29.443581  -26.579767   13.155642   

                                EEG Fz      EEG F4      EEG F8 

`SE038_a2_df` went from [1606399 rows] to [468479 rows]

#### Create Bipolar Channel Pairs

In [75]:
#SE008_a2_df = create_bipolar_channels(SE008_a2_df)
SE021_a2_df = create_bipolar_channels(SE021_a2_df)
#SE026_a2_df = create_bipolar_channels(SE026_a2_df)
SE038_a2_df = create_bipolar_channels(SE038_a2_df)

In [76]:
#print(f"SE008_a2_df",SE008_a2_df.head())
print(f"SE021_a2_df",SE021_a2_df.head())
#print(f"SE026_a2_df",SE026_a2_df.head())
print(f"SE038_a2_df",SE038_a2_df.head())

SE021_a2_df                                Fp1-F7      F7-T3      T3-T5      Fp1-F3  \
2015-09-01 16:06:27.000000  34.186771  62.287937 -63.898834  141.758759   
2015-09-01 16:06:27.003906  35.260700  60.856033 -67.836578  140.863815   
2015-09-01 16:06:27.007812  34.902725  59.066147 -69.626465  139.789886   
2015-09-01 16:06:27.011718  35.260704  56.918289 -68.731522  138.536957   
2015-09-01 16:06:27.015625  36.155640  56.202335 -65.509727  139.431915   

                                 F3-C3       C3-P3      P3-O1       Fz-Cz  \
2015-09-01 16:06:27.000000 -108.645912  150.887161 -32.575874  141.937744   
2015-09-01 16:06:27.003906 -109.361870  148.560303 -31.322960  138.536957   
2015-09-01 16:06:27.007812 -110.077820  146.412445 -30.785988  138.000000   
2015-09-01 16:06:27.011718 -109.540855  147.307404 -32.396889  137.821014   
2015-09-01 16:06:27.015625 -108.287941  148.560303 -33.470818  138.536957   

                                Cz-Pz      Fp2-F4      F4-C4      C4-P4  \

#### Create Ground Truth Labels
Using `_annotations.txt` files for all patients

#### TO DO: Missing Ground Truth Labels
for remaining (Andreas) annotated files
- [X] ~~Convert SE038_annotated2.edf~~
- [ ] Convert SE008_annotated2_annotations.txt
- [ ] Convert SE026_annotated2_annotations.txt

In [77]:
# SE008_a2_annotations = pd.read_csv(RAW_KISPI_DATA_FOLDER / "SE008_annotated2_annotations.txt")
SE021_a2_annotations = pd.read_csv(RAW_KISPI_DATA_FOLDER / "SE021_annotated2_annotations.txt")
#SE026_a2_annotations = pd.read_csv(RAW_KISPI_DATA_FOLDER / "SE026_annotated2_annotations.txt") # FILE CURRENTLY DOESN'T EXIST  
SE038_a2_annotations = pd.read_csv(RAW_KISPI_DATA_FOLDER / "SE038_annotated2_annotations.txt")

##### SE008_a2 Ground Truth Labels
Create `SE008_a2_ground_truth_df`

In [78]:
# SE008_a2_annotations["Onset"] = pd.to_datetime(SE008_a2_annotations["Onset"], format="%Y-%m-%dT%H:%M:%S.%f")
# 
# SE008_a2_timestamps = SE008_a2_df.index
# SE008_a2_ground_truth_df = pd.DataFrame(
#     {"timestamp" : SE008_a1_timestamps, "ground_truth" : 0}
#     )
# 
# SE008_a2_ground_truth_df["timestamp"] = pd.to_datetime(SE008_a2_ground_truth_df["timestamp"], format="%Y-%m-%dT%H:%M:%S.%f")
# 
# for index, row in SE008_a2_annotations.iterrows():
#     if row["Annotation"] == "Burst starts":
#         start_time = row["Onset"]
#         try:
#             # find the next row where the annotation is "Burst end"
#             end_time = SE008_a2_annotations.loc[
#                 (SE008_a1_annotations["Annotation"] == "Burst end") &
#                 (SE008_a1_annotations.index > index), "Onset"
#             ].iloc[0]
#         
#             SE008_a2_ground_truth_df.loc[
#                 (SE008_a2_ground_truth_df["timestamp"] >= start_time) 
#                 & (SE008_a2_ground_truth_df["timestamp"] <= end_time), 
#                 "ground_truth",] = 1
#         except IndexError:
#             print(f"Warning: No matching 'Burst end' annotation found for 'Burst starts' at {start_time}")

In [79]:
# SE008_a1_ground_truth_df.head()

In [80]:
# print(SE008_a2_ground_truth_df.shape)
# print(SE008_a2_df.shape)

##### SE021_a2 Ground Truth Labels
Create `SE021_a2_ground_truth_df`

In [81]:
SE021_a2_annotations.head()

,Onset,Duration,Annotation
0,2015-09-01T16:09:25.5570000,0.499,Burst
1,2015-09-01T16:09:45.2990000,0.525,Burst
2,2015-09-01T16:09:48.3110000,0.511,Burst
3,2015-09-01T16:09:52.3260000,0.550,Burst
4,2015-09-01T16:10:42.1490000,0.466,Burst


In [82]:
SE021_a2_df.index = SE021_a2_df.index.tz_localize(None)

In [83]:
SE021_a2_annotations["Onset"] = pd.to_datetime(SE021_a2_annotations["Onset"], format="%Y-%m-%dT%H:%M:%S.%f")

SE021_a2_timestamps = SE021_a2_df.index

SE021_a2_ground_truth_df = pd.DataFrame(
    {"timestamp" : SE021_a2_timestamps, "ground_truth" : 0}
    )

SE021_a2_ground_truth_df["timestamp"] = pd.to_datetime(SE021_a2_ground_truth_df["timestamp"], format="%Y-%m-%dT%H:%M:%S.%f")

# label "burst" or "Burst Suppression" annotations with duration
for index, row in SE021_a2_annotations.iterrows():
    if row["Annotation"] == "burst" or "Burst Suppression" and pd.notna(row["Duration"]):
        start_time = row["Onset"]
        end_time = start_time + pd.to_timedelta(row["Duration"], unit='s')  # calculate end_time using duration
        
        print(f"Labeling from {start_time} to {end_time}")

        SE021_a2_ground_truth_df.loc[
            (SE021_a2_ground_truth_df["timestamp"] >= start_time)
            & (SE021_a2_ground_truth_df["timestamp"] <= end_time),
            "ground_truth",
        ] = 1

Labeling from 2015-09-01 16:09:25.557000 to 2015-09-01 16:09:26.056000
Labeling from 2015-09-01 16:09:45.299000 to 2015-09-01 16:09:45.824000
Labeling from 2015-09-01 16:09:48.311000 to 2015-09-01 16:09:48.822000
Labeling from 2015-09-01 16:09:52.326000 to 2015-09-01 16:09:52.876000
Labeling from 2015-09-01 16:10:42.149000 to 2015-09-01 16:10:42.615000
Labeling from 2015-09-01 16:11:05.880000 to 2015-09-01 16:11:06.385000
Labeling from 2015-09-01 16:12:06.905000 to 2015-09-01 16:12:07.357000
Labeling from 2015-09-01 16:12:29.682000 to 2015-09-01 16:12:30.226000
Labeling from 2015-09-01 16:12:34.741000 to 2015-09-01 16:12:35.226000
Labeling from 2015-09-01 16:13:23.933000 to 2015-09-01 16:13:24.457000
Labeling from 2015-09-01 16:13:34.051000 to 2015-09-01 16:13:34.654000
Labeling from 2015-09-01 16:14:14.651000 to 2015-09-01 16:14:15.176000
Labeling from 2015-09-01 16:14:23.448000 to 2015-09-01 16:14:23.835000
Labeling from 2015-09-01 16:14:34.990000 to 2015-09-01 16:14:35.468000
Labeli

In [84]:
SE021_a2_ground_truth_df.head(-1)

,timestamp,ground_truth
0,2015-09-01 16:06:27.000000,0
1,2015-09-01 16:06:27.003906,0
2,2015-09-01 16:06:27.007812,0
3,2015-09-01 16:06:27.011718,0
4,2015-09-01 16:06:27.015625,0
...,...,...
368634,2015-09-01 16:30:26.976562,0
368635,2015-09-01 16:30:26.980468,0
368636,2015-09-01 16:30:26.984375,0
368637,2015-09-01 16:30:26.988281,0


In [85]:
print(SE021_a2_ground_truth_df.shape)
print(SE021_a2_df.shape)

(368640, 2)
(368640, 17)


In [86]:
SE021_a2_ground_truth_df.to_csv(
    RAW_KISPI_DATA_FOLDER / "SE021_a2_ground_truth.csv",
    date_format="%Y-%m-%d %H:%M:%S.%f")

##### SE026_a2 Ground Truth Labels
Create `SE026_a2_ground_truth_df`

In [87]:
# SE026_a2_annotations.head(-1)

In [88]:
# SE026_a2_df.index = SE026_a2_df.index.tz_localize(None)

In [89]:
# SE026_a2_annotations["Onset"] = pd.to_datetime(SE026_a2_annotations["Onset"], format="%Y-%m-%dT%H:%M:%S.%f")
# 
# SE026_a2_timestamps = SE026_a2_df.index
# 
# SE026_a2_ground_truth_df = pd.DataFrame(
#     {"timestamp" : SE026_a2_timestamps, "ground_truth" : 0}
#     )
# 
# SE026_a2_ground_truth_df["timestamp"] = pd.to_datetime(SE026_a2_ground_truth_df["timestamp"], format="%Y-%m-%dT%H:%M:%S.%f")
# 
# burst_count = 0
# for index, row in SE026_a2_annotations.iterrows():
#     if row["Annotation"] == "burst":
#         burst_count += 1
#         if burst_count % 2 == 0:  # Check if it's an even count (should be a closing 'burst')
#             end_time = row["Onset"]
#             
#             SE026_a2_ground_truth_df.loc[
#                 (SE026_a2_ground_truth_df["timestamp"] >= start_time)
#                 & (SE026_a2_ground_truth_df["timestamp"] <= end_time),
#                 "ground_truth",
#             ] = 1
#         else:
#             start_time = row["Onset"]
# 
# if burst_count % 2 != 0:
#     print(f"Warning: Unclosed 'burst' at {start_time}")

In [90]:
# SE026_a2_ground_truth_df.head()

In [91]:
# print(SE026_a2_ground_truth_df.shape)
# print(SE026_a2_df.shape)

##### SE038_a2 Ground Truth Labels
Create `SE038_a2_ground_truth_df`

In [92]:
SE038_a2_annotations.head(-1)

,Onset,Duration,Annotation
0,2018-05-23T11:06:00.5960000,0.460,Burst Suppression
1,2018-05-23T11:06:05.2390000,0.459,Burst Suppression
2,2018-05-23T11:06:21.2620000,0.387,Burst Suppression
3,2018-05-23T11:06:27.7700000,0.584,Burst Suppression
4,2018-05-23T11:06:31.1030000,0.452,Burst
...,...,...,...
224,2018-05-23T11:34:40.5140000,0.335,Burst
225,2018-05-23T11:34:48.1240000,0.460,Burst
226,2018-05-23T11:34:54.0000000,0.282,Burst
227,2018-05-23T11:34:59.3960000,0.335,Burst


In [93]:
SE038_a2_annotations["Onset"] = pd.to_datetime(SE038_a2_annotations["Onset"], format="%Y-%m-%dT%H:%M:%S.%f")

SE038_a2_timestamps = SE038_a2_df.index

SE038_a2_ground_truth_df = pd.DataFrame(
    {"timestamp" : SE038_a2_timestamps, "ground_truth" : 0}
    )

SE038_a2_ground_truth_df["timestamp"] = pd.to_datetime(SE038_a2_ground_truth_df["timestamp"], format="%Y-%m-%dT%H:%M:%S.%f")

# label "burst" or "Burst Suppression" annotations with duration
for index, row in SE038_a2_annotations.iterrows():
    if row["Annotation"] == "burst" or "Burst Suppression" and pd.notna(row["Duration"]):
        start_time = row["Onset"]
        end_time = start_time + pd.to_timedelta(row["Duration"], unit='s')  # calculate end_time using duration
        
        print(f"Labeling from {start_time} to {end_time}")

        SE038_a2_ground_truth_df.loc[
            (SE038_a2_ground_truth_df["timestamp"] >= start_time)
            & (SE038_a2_ground_truth_df["timestamp"] <= end_time),
            "ground_truth",
        ] = 1

Labeling from 2018-05-23 11:06:00.596000 to 2018-05-23 11:06:01.056000
Labeling from 2018-05-23 11:06:05.239000 to 2018-05-23 11:06:05.698000
Labeling from 2018-05-23 11:06:21.262000 to 2018-05-23 11:06:21.649000
Labeling from 2018-05-23 11:06:27.770000 to 2018-05-23 11:06:28.354000
Labeling from 2018-05-23 11:06:31.103000 to 2018-05-23 11:06:31.555000
Labeling from 2018-05-23 11:06:38.170000 to 2018-05-23 11:06:38.577000
Labeling from 2018-05-23 11:06:42.795000 to 2018-05-23 11:06:43.109000
Labeling from 2018-05-23 11:06:49.039000 to 2018-05-23 11:06:49.374000
Labeling from 2018-05-23 11:06:53.330000 to 2018-05-23 11:06:53.664000
Labeling from 2018-05-23 11:06:58.546000 to 2018-05-23 11:06:58.821000
Labeling from 2018-05-23 11:07:02.878000 to 2018-05-23 11:07:03.258000
Labeling from 2018-05-23 11:07:06.880000 to 2018-05-23 11:07:07.227000
Labeling from 2018-05-23 11:07:10.533000 to 2018-05-23 11:07:10.841000
Labeling from 2018-05-23 11:07:15.738000 to 2018-05-23 11:07:16.151000
Labeli

In [94]:
SE038_a2_ground_truth_df.head(-1)

,timestamp,ground_truth
0,2018-05-23 11:05:57.000000,0
1,2018-05-23 11:05:57.003906,0
2,2018-05-23 11:05:57.007812,0
3,2018-05-23 11:05:57.011718,0
4,2018-05-23 11:05:57.015625,0
...,...,...
468474,2018-05-23 11:36:26.976562,0
468475,2018-05-23 11:36:26.980468,0
468476,2018-05-23 11:36:26.984375,0
468477,2018-05-23 11:36:26.988281,0


In [95]:
print(SE038_a2_ground_truth_df.shape)
print(SE038_a2_df.shape)

(468480, 2)
(468480, 17)


In [96]:
SE038_a2_ground_truth_df.to_csv(
    RAW_KISPI_DATA_FOLDER / "SE038_a2_ground_truth.csv",
    date_format="%Y-%m-%d %H:%M:%S.%f")

### Merge Annotator-2 Dataframes

In [97]:
#bipolar_SE008_a2 = SE008_a2_df.copy()    
bipolar_SE021_a2 = SE021_a2_df.copy()    
#bipolar_SE026_a2 = SE026_a2_df.copy()    
bipolar_SE038_a2 = SE038_a2_df.copy()

In [98]:
#bipolar_SE008_a2_merged = pd.merge(bipolar_SE008_a2, SE008_a2_ground_truth_df, left_index=True, right_on="timestamp") 
bipolar_SE021_a2_merged = pd.merge(bipolar_SE021_a2, SE021_a2_ground_truth_df, left_index=True, right_on="timestamp")
#bipolar_SE026_a2_merged = pd.merge(bipolar_SE026_a2, SE026_a2_ground_truth_df, left_index=True, right_on="timestamp")
bipolar_SE038_a2_merged = pd.merge(bipolar_SE038_a2, SE038_a2_ground_truth_df, left_index=True, right_on="timestamp")

In [99]:
#print(bipolar_SE008_a2_merged.shape)
print(bipolar_SE021_a2_merged.shape)
#print(bipolar_SE026_a2_merged.shape)
print(bipolar_SE038_a2_merged.shape)

(368640, 19)
(468480, 19)


#### Drop Datetime Index
We now drop the current datatime index.

This will allows us to create our `data_learning` (Train) and `data_testing` (Test) datasets.

In [100]:
#bipolar_SE008_a2_merged.reset_index(drop=True, inplace=True)
#print(bipolar_SE008_a2_merged.head(-1))

bipolar_SE021_a2_merged.reset_index(drop=True, inplace=True)
print(bipolar_SE021_a2_merged.head(-1))

#bipolar_SE026_a2_merged.reset_index(drop=True, inplace=True)

bipolar_SE038_a2_merged.reset_index(drop=True, inplace=True)

           Fp1-F7      F7-T3      T3-T5      Fp1-F3       F3-C3       C3-P3  \
0       34.186771  62.287937 -63.898834  141.758759 -108.645912  150.887161   
1       35.260700  60.856033 -67.836578  140.863815 -109.361870  148.560303   
2       34.902725  59.066147 -69.626465  139.789886 -110.077820  146.412445   
3       35.260704  56.918289 -68.731522  138.536957 -109.540855  147.307404   
4       36.155640  56.202335 -65.509727  139.431915 -108.287941  148.560303   
...           ...        ...        ...         ...         ...         ...   
368634  64.077820  40.451363 -72.132294  130.303497 -151.961090  175.229584   
368635  63.361866  41.346306 -70.700394  130.482498 -149.634247  173.618683   
368636  61.750973  42.062252 -69.626457  130.303497 -146.233459  172.007782   
368637  60.319065  40.272377 -68.373543  130.482483 -147.307388  171.291824   
368638  59.603111  38.482491 -66.225677  131.198441 -149.634232  172.365753   

            P3-O1       Fz-Cz      Cz-Pz      Fp2-F

#### Drop `Timestamp` column

In [101]:
#bipolar_SE008_a2_merged.drop(columns="timestamp", inplace=True)
#print(bipolar_SE008_a2_merged.head(-1))

bipolar_SE021_a2_merged.drop(columns="timestamp", inplace=True)
print(bipolar_SE021_a2_merged.head(-1))

#bipolar_SE026_a2_merged.drop(columns="timestamp", inplace=True) 
#print(bipolar_SE026_a2_merged.head(-1)) 

bipolar_SE038_a2_merged.drop(columns="timestamp", inplace=True)
print(bipolar_SE038_a2_merged.head(-1))

           Fp1-F7      F7-T3      T3-T5      Fp1-F3       F3-C3       C3-P3  \
0       34.186771  62.287937 -63.898834  141.758759 -108.645912  150.887161   
1       35.260700  60.856033 -67.836578  140.863815 -109.361870  148.560303   
2       34.902725  59.066147 -69.626465  139.789886 -110.077820  146.412445   
3       35.260704  56.918289 -68.731522  138.536957 -109.540855  147.307404   
4       36.155640  56.202335 -65.509727  139.431915 -108.287941  148.560303   
...           ...        ...        ...         ...         ...         ...   
368634  64.077820  40.451363 -72.132294  130.303497 -151.961090  175.229584   
368635  63.361866  41.346306 -70.700394  130.482498 -149.634247  173.618683   
368636  61.750973  42.062252 -69.626457  130.303497 -146.233459  172.007782   
368637  60.319065  40.272377 -68.373543  130.482483 -147.307388  171.291824   
368638  59.603111  38.482491 -66.225677  131.198441 -149.634232  172.365753   

            P3-O1       Fz-Cz      Cz-Pz      Fp2-F

## LOAD

### Create Parquet Files
From transformed dataframes

In [102]:
bipolar_SE008_a1_merged.to_parquet(PROCESSED_KISPI_DATA_FOLDER / "SE008_a1_merged.parquet")

bipolar_SE021_a1_merged.to_parquet(PROCESSED_KISPI_DATA_FOLDER / "SE021_a1_merged.parquet")

bipolar_SE026_a1_merged.to_parquet(PROCESSED_KISPI_DATA_FOLDER / "SE026_a1_merged.parquet")

bipolar_SE038_a1_merged.to_parquet(PROCESSED_KISPI_DATA_FOLDER / "SE038_a1_merged.parquet")

In [103]:
#bipolar_SE008_a2_merged.to_parquet(PROCESSED_KISPI_DATA_FOLDER / "SE008_a2_merged.parquet")

bipolar_SE021_a2_merged.to_parquet(PROCESSED_KISPI_DATA_FOLDER / "SE021_a2_merged.parquet")

#bipolar_SE026_a2_merged.to_parquet(PROCESSED_KISPI_DATA_FOLDER / "SE026_a2_merged.parquet")

bipolar_SE038_a2_merged.to_parquet(PROCESSED_KISPI_DATA_FOLDER / "SE038_a2_merged.parquet")

### Create data_attributes.csv file for kispi data  
- [x] with the following code make a data_attributes.csv file

In [2]:
def create_data_attributes_csv(parquet_files, output_folder):
    """
    Creates a CSV file named 'data_attributes_kispi.csv' from multiple Parquet files
    containing patient data.

    Args:
        parquet_files: List of paths to the Parquet files.
        output_folder: Path to the output folder.
    """
    data_attributes_list = []

    for i, parquet_file in enumerate(parquet_files):
        reader = ParquetReader(parquet_file)
        file_header = reader.get_file_header()
        signal_headers = reader.get_signal_headers()

        try:
            patient_id_parts = file_header['patientcode'].split(' ')[0] + "_" + file_header['patientcode'].split(' ')[1]
        except KeyError:
            print("KeyError: 'patientcode' key not found in file header.")
            patient_id_parts = parquet_file.stem
        unique_patient_id = patient_id_parts
        start_datetime = file_header['startdate'].strftime('%Y-%m-%d %H:%M:%S')

        # get end sample and sample rate
        df = pd.read_parquet(parquet_file)
        end_sample = df.index[-1]  # get last datetime index
        sample_rate = signal_headers[0]['sample_rate']

        # create a dictionary with patient ID, start date, end sample, and sample rate
        data_attributes = {
            'unique_patient_id': unique_patient_id,
            'patient_id': f"P{i + 1}",
            'start_datetime': start_datetime,
            'end_sample': end_sample,
            'sample_rate': sample_rate
        }
        data_attributes_list.append(data_attributes)

    # Create the output directory if it doesn't exist
    output_dir = os.path.dirname(output_folder)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # create a DataFrame from the list of dictionaries
    data_attributes_df = pd.DataFrame(data_attributes_list)

    try:
        data_attributes_df.to_csv(output_folder, index=False)
    except PermissionError:
        print(f"Permission denied: Unable to write to {output_dir}")

    print(f"Data attributes saved to {output_folder}")

In [3]:

# Define the input and output folders
input_folder = PROCESSED_KISPI_DATA_FOLDER
output_folder = PROCESSED_KISPI_DATA_FOLDER / Path("data_attributes_kispi.csv")

parequet_filenames = [input_folder / filename for filename in os.listdir(input_folder) if  filename.endswith('.parquet') - filename.__contains__('merged')]

parequet_filenames

[WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE008_a1.parquet'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE021_a1.parquet'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE021_a2.parquet'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE026_a1.parquet'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE038_a1.parquet'),
 WindowsPath('C:/Users/c_arz/Documents/KISPI/Burst_Suppression_Project/Thesis/UnsuperDL-EEG-BSUPP/data/processed_kispi/SE038_a2.parquet')]

In [6]:
create_data_attributes_csv(parequet_filenames, output_folder)

Data attributes saved to C:\Users\c_arz\Documents\KISPI\Burst_Suppression_Project\Thesis\UnsuperDL-EEG-BSUPP\data\processed_kispi\data_attributes_kispi.csv
